# E-commerce Intelligence & Decision System
## Integrated Day 1–2 Notebook

**Person A — Analytics Engine**
- Cleaning and transaction preparation
- Revenue, margin, product profitability
- RFM segmentation
- Cohort retention
- Price elasticity / what-if analysis
- Product contribution and discontinuation recommendations
- Shared data contract

**Person B — NEXUS**
- Streamlit dashboard scaffold
- Real analytics connected to the dashboard
- AI Analyst query routing and Gemini narration
- Clean separation between analytics and presentation


In [ ]:
!pip install -q openpyxl plotly streamlit google-genai

import os
import json
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully!")


Libraries loaded successfully!


## 1. Load and clean the dataset

In [ ]:
FILE_PATH = "/content/Online Retail.xlsx"

df_raw = pd.read_excel(FILE_PATH)

print("Raw data shape:", df_raw.shape)


Raw data shape: (541909, 8)


In [ ]:
def clean_transactions(df):
    data = df.copy()

    # 1. Remove exact duplicate rows
    data = data.drop_duplicates()

    # 2. Remove rows without CustomerID
    data = data.dropna(subset=["CustomerID"])

    # 3. Parse date
    data["InvoiceDate"] = pd.to_datetime(
        data["InvoiceDate"],
        errors="coerce"
    )

    # 4. Remove rows where date couldn't be parsed
    data = data.dropna(subset=["InvoiceDate"])

    # 5. Remove returns / negative quantities
    data = data[data["Quantity"] > 0]

    # 6. Remove zero or negative prices
    data = data[data["UnitPrice"] > 0]

    # 7. Convert CustomerID to integer
    data["CustomerID"] = data["CustomerID"].astype(int)

    return data.reset_index(drop=True)


df_clean = clean_transactions(df_raw)

df_clean["Revenue"] = (
    df_clean["Quantity"] * df_clean["UnitPrice"]
)

print("Raw shape:  ", df_raw.shape)
print("Clean shape:", df_clean.shape)


Raw shape:   (541909, 8)
Clean shape: (392692, 9)


## 2. Revenue and profitability

In [ ]:
COST_RATE = 0.60

df_clean["Estimated_COGS"] = (
    df_clean["Revenue"] * COST_RATE
)

df_clean["Gross_Profit"] = (
    df_clean["Revenue"] - df_clean["Estimated_COGS"]
)

df_clean["Gross_Margin"] = (
    df_clean["Gross_Profit"] / df_clean["Revenue"]
)


In [ ]:
def revenue_trend(df):
    data = df.copy()

    data["Month"] = (
        data["InvoiceDate"]
        .dt.to_period("M")
        .astype(str)
    )

    result = (
        data.groupby("Month")
        .agg(
            Revenue=("Revenue", "sum"),
            Orders=("InvoiceNo", "nunique"),
            Customers=("CustomerID", "nunique")
        )
        .reset_index()
    )

    return result


revenue_df = revenue_trend(df_clean)


In [ ]:
def gross_margin_summary(df):
    total_revenue = df["Revenue"].sum()
    total_cogs = df["Estimated_COGS"].sum()
    gross_profit = total_revenue - total_cogs

    gross_margin = (
        gross_profit / total_revenue
        if total_revenue != 0
        else 0
    )

    return {
        "revenue": total_revenue,
        "cogs": total_cogs,
        "gross_profit": gross_profit,
        "gross_margin": gross_margin
    }


margin_dict = gross_margin_summary(df_clean)


In [ ]:
def product_profitability(df):
    result = (
        df.groupby("Description")
        .agg(
            Revenue=("Revenue", "sum"),
            COGS=("Estimated_COGS", "sum"),
            Gross_Profit=("Gross_Profit", "sum")
        )
        .reset_index()
    )

    result["Gross_Margin"] = (
        result["Gross_Profit"] / result["Revenue"]
    )

    return result.sort_values(
        "Revenue",
        ascending=False
    )


product_df = product_profitability(df_clean)


## 3. RFM customer segmentation

In [ ]:
def build_rfm(df):
    snapshot_date = (
        df["InvoiceDate"].max()
        + pd.Timedelta(days=1)
    )

    rfm = (
        df.groupby("CustomerID")
        .agg(
            Recency=(
                "InvoiceDate",
                lambda x: (snapshot_date - x.max()).days
            ),
            Frequency=("InvoiceNo", "nunique"),
            Monetary=("Revenue", "sum")
        )
        .reset_index()
    )

    return rfm


In [ ]:
def add_rfm_scores(rfm):
    data = rfm.copy()

    data["R_Score"] = pd.qcut(
        data["Recency"],
        4,
        labels=[4, 3, 2, 1]
    )

    data["F_Score"] = pd.qcut(
        data["Frequency"].rank(method="first"),
        4,
        labels=[1, 2, 3, 4]
    )

    data["M_Score"] = pd.qcut(
        data["Monetary"],
        4,
        labels=[1, 2, 3, 4]
    )

    return data


In [ ]:
def assign_segment(row):

    r = int(row["R_Score"])
    f = int(row["F_Score"])
    m = int(row["M_Score"])

    if r >= 3 and f >= 3 and m >= 3:
        return "Champions"

    elif r >= 3 and f >= 2:
        return "Loyal Customers"

    elif r <= 2 and f >= 3:
        return "At Risk"

    elif r <= 2 and f <= 2:
        return "Lost Customers"

    else:
        return "Potential Customers"


rfm_df = add_rfm_scores(build_rfm(df_clean))
rfm_df["Segment"] = rfm_df.apply(
    assign_segment,
    axis=1
)


## 4. Cohort retention

In [ ]:
def build_cohort_data(df):

    data = df.copy()

    data["Purchase_Month"] = (
        data["InvoiceDate"]
        .dt.to_period("M")
    )

    first_purchase = (
        data.groupby("CustomerID")["Purchase_Month"]
        .min()
        .rename("Cohort_Month")
    )

    data = data.merge(
        first_purchase,
        on="CustomerID",
        how="left"
    )

    return data


def add_cohort_period(df):

    data = df.copy()

    data["Cohort_Period"] = (
        (data["Purchase_Month"].dt.year -
         data["Cohort_Month"].dt.year) * 12
        +
        (data["Purchase_Month"].dt.month -
         data["Cohort_Month"].dt.month)
        + 1
    )

    return data


def build_retention_table(df):

    data = df.copy()

    cohort_counts = (
        data.groupby(
            ["Cohort_Month", "Cohort_Period"]
        )["CustomerID"]
        .nunique()
        .reset_index()
    )

    cohort_pivot = cohort_counts.pivot(
        index="Cohort_Month",
        columns="Cohort_Period",
        values="CustomerID"
    )

    cohort_size = cohort_pivot.iloc[:, 0]

    retention = cohort_pivot.divide(
        cohort_size,
        axis=0
    ) * 100

    return retention


cohort_data = build_cohort_data(df_clean)
cohort_data = add_cohort_period(cohort_data)
retention_df = build_retention_table(cohort_data)


## 5. Decision / simulation layer

In [ ]:
def prepare_price_quantity_data(df):
    data = df.copy()

    data["Month"] = (
        data["InvoiceDate"]
        .dt.to_period("M")
        .astype(str)
    )

    result = (
        data.groupby(["StockCode", "Description", "Month"])
        .agg(
            Avg_Price=("UnitPrice", "mean"),
            Quantity=("Quantity", "sum"),
            Revenue=("Revenue", "sum")
        )
        .reset_index()
    )

    return result


def estimate_price_elasticity(df, min_observations=6):
    results = []

    for product, group in df.groupby("StockCode"):

        data = group[
            (group["Avg_Price"] > 0) &
            (group["Quantity"] > 0)
        ].copy()

        if len(data) < min_observations:
            continue

        x = np.log(data["Avg_Price"])
        y = np.log(data["Quantity"])

        slope, intercept = np.polyfit(x, y, 1)

        predicted = intercept + slope * x

        ss_res = ((y - predicted) ** 2).sum()
        ss_tot = ((y - y.mean()) ** 2).sum()

        r_squared = (
            1 - ss_res / ss_tot
            if ss_tot != 0
            else np.nan
        )

        results.append({
            "StockCode": product,
            "Description": data["Description"].iloc[0],
            "Elasticity": slope,
            "R_Squared": r_squared,
            "Observations": len(data),
            "Avg_Price": data["Avg_Price"].mean(),
            "Avg_Monthly_Quantity": data["Quantity"].mean()
        })

    return pd.DataFrame(results)


def classify_elasticity(elasticity):

    if pd.isna(elasticity):
        return "Insufficient Data"

    if elasticity < -1:
        return "Elastic"

    elif elasticity > -1 and elasticity < 0:
        return "Inelastic"

    elif elasticity == -1:
        return "Unit Elastic"

    else:
        return "Positive / Anomalous"


def price_what_if(
    elasticity,
    current_price,
    current_quantity,
    price_change_pct
):
    new_price = current_price * (
        1 + price_change_pct / 100
    )

    predicted_quantity = (
        current_quantity *
        (new_price / current_price) ** elasticity
    )

    current_revenue = (
        current_price * current_quantity
    )

    predicted_revenue = (
        new_price * predicted_quantity
    )

    return {
        "Current_Price": current_price,
        "New_Price": new_price,
        "Current_Quantity": current_quantity,
        "Predicted_Quantity": predicted_quantity,
        "Current_Revenue": current_revenue,
        "Predicted_Revenue": predicted_revenue,
        "Revenue_Change_Pct": (
            (predicted_revenue / current_revenue) - 1
        ) * 100
    }


def price_scenario_table(
    elasticity,
    current_price,
    current_quantity,
    changes=[-20, -10, -5, 0, 5, 10, 20]
):

    results = []

    for change in changes:

        result = price_what_if(
            elasticity,
            current_price,
            current_quantity,
            change
        )

        result["Price_Change_Pct"] = change

        results.append(result)

    return pd.DataFrame(results)


def product_contribution(df):

    result = (
        df.groupby(["StockCode", "Description"])
        .agg(
            Revenue=("Revenue", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("InvoiceNo", "nunique"),
            Gross_Profit=("Gross_Profit", "sum")
        )
        .reset_index()
    )

    result["Gross_Margin"] = (
        result["Gross_Profit"] /
        result["Revenue"]
    )

    return result


def flag_discontinuation_candidates(df):

    data = df.copy()

    low_volume_threshold = data["Quantity"].quantile(0.25)
    low_margin_threshold = data["Gross_Margin"].quantile(0.25)

    data["Low_Volume"] = (
        data["Quantity"] <= low_volume_threshold
    )

    data["Low_Margin"] = (
        data["Gross_Margin"] <= low_margin_threshold
    )

    data["Discontinue_Flag"] = (
        data["Low_Volume"] &
        data["Low_Margin"]
    )

    return data


def assign_product_action(row):

    if row["Low_Volume"] and row["Low_Margin"]:
        return "Discontinue Candidate"

    elif row["Low_Volume"]:
        return "Review Demand"

    elif row["Low_Margin"]:
        return "Review Pricing/Cost"

    else:
        return "Keep"


price_qty_df = prepare_price_quantity_data(df_clean)
elasticity_df = estimate_price_elasticity(price_qty_df)

elasticity_df["Elasticity_Type"] = (
    elasticity_df["Elasticity"]
    .apply(classify_elasticity)
)

contribution_df = product_contribution(df_clean)

discontinuation_df = flag_discontinuation_candidates(
    contribution_df
)

discontinuation_df["Recommended_Action"] = (
    discontinuation_df
    .apply(assign_product_action, axis=1)
)


/tmp/ipykernel_7617/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_7617/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_7617/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_7617/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_7617/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_7617/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_7617/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_7617/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)


## 6. Shared data contract

This is the hand-off between **Person A's analytics engine** and **Person B's NEXUS dashboard / AI Analyst**.


In [ ]:
data_contract = {
    "clean_transactions": df_clean,
    "revenue_trend": revenue_df,
    "product_profitability": product_df,
    "rfm_customers": rfm_df,
    "retention": retention_df,
    "gross_margin": margin_dict,
    "price_elasticity": elasticity_df,
    "product_discontinuation": discontinuation_df
}

print("=== DATA CONTRACT ===")
for name, value in data_contract.items():
    if hasattr(value, "shape"):
        print(f"{name}: {value.shape}")
    else:
        print(f"{name}: ready")


=== DATA CONTRACT ===
clean_transactions: (392692, 12)
revenue_trend: (13, 4)
product_profitability: (3877, 5)
rfm_customers: (4338, 8)
retention: (13, 13)
gross_margin: ready
price_elasticity: (2567, 8)
product_discontinuation: (3897, 11)


## 7. Export the shared analytics outputs

In [ ]:
OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_clean.to_csv(f"{OUTPUT_DIR}/clean_transactions.csv", index=False)
revenue_df.to_csv(f"{OUTPUT_DIR}/revenue_trend.csv", index=False)
product_df.to_csv(f"{OUTPUT_DIR}/product_profitability.csv", index=False)
rfm_df.to_csv(f"{OUTPUT_DIR}/rfm_customers.csv", index=False)
retention_df.to_csv(f"{OUTPUT_DIR}/retention.csv")
elasticity_df.to_csv(f"{OUTPUT_DIR}/price_elasticity.csv", index=False)
discontinuation_df.to_csv(
    f"{OUTPUT_DIR}/product_discontinuation.csv",
    index=False
)

with open(f"{OUTPUT_DIR}/gross_margin.json", "w") as f:
    json.dump(margin_dict, f, indent=4)

print("Analytics outputs exported successfully.")


Analytics outputs exported successfully.


## 8. NEXUS backend

The following cell creates the shared backend used by the Streamlit application.  
It keeps Person A's analytics intact and adds Person B's query routing + AI narration layer.


In [ ]:
%%writefile backend.py
import os
import pandas as pd
import numpy as np

from google import genai


def load_and_clean_data(file_path):
    df = pd.read_excel(file_path)

    df = df.drop_duplicates()
    df = df.dropna(subset=["CustomerID"])

    df["InvoiceDate"] = pd.to_datetime(
        df["InvoiceDate"],
        errors="coerce"
    )

    df = df.dropna(subset=["InvoiceDate"])
    df = df[df["Quantity"] > 0]
    df = df[df["UnitPrice"] > 0]

    df["CustomerID"] = df["CustomerID"].astype(int)

    df["Revenue"] = (
        df["Quantity"] * df["UnitPrice"]
    )

    df["Estimated_COGS"] = (
        df["Revenue"] * 0.60
    )

    df["Gross_Profit"] = (
        df["Revenue"] - df["Estimated_COGS"]
    )

    df["Gross_Margin"] = (
        df["Gross_Profit"] / df["Revenue"]
    )

    return df.reset_index(drop=True)


def build_revenue_data(df):
    data = df.copy()

    data["Month"] = (
        data["InvoiceDate"]
        .dt.to_period("M")
        .astype(str)
    )

    result = (
        data.groupby("Month")
        .agg(
            Revenue=("Revenue", "sum"),
            Orders=("InvoiceNo", "nunique"),
            Customers=("CustomerID", "nunique")
        )
        .reset_index()
    )

    return result


def build_rfm(df):
    snapshot_date = (
        df["InvoiceDate"].max()
        + pd.Timedelta(days=1)
    )

    rfm = (
        df.groupby("CustomerID")
        .agg(
            Recency=(
                "InvoiceDate",
                lambda x: (snapshot_date - x.max()).days
            ),
            Frequency=("InvoiceNo", "nunique"),
            Monetary=("Revenue", "sum")
        )
        .reset_index()
    )

    rfm["R_Score"] = pd.qcut(
        rfm["Recency"],
        4,
        labels=[4, 3, 2, 1]
    )

    rfm["F_Score"] = pd.qcut(
        rfm["Frequency"].rank(method="first"),
        4,
        labels=[1, 2, 3, 4]
    )

    rfm["M_Score"] = pd.qcut(
        rfm["Monetary"],
        4,
        labels=[1, 2, 3, 4]
    )

    def segment(row):
        r = int(row["R_Score"])
        f = int(row["F_Score"])
        m = int(row["M_Score"])

        if r >= 3 and f >= 3 and m >= 3:
            return "Champions"
        elif r >= 3 and f >= 2:
            return "Loyal Customers"
        elif r <= 2 and f >= 3:
            return "At Risk"
        elif r <= 2 and f <= 2:
            return "Lost Customers"
        return "Potential Customers"

    rfm["Segment"] = rfm.apply(segment, axis=1)

    return rfm


def build_cohort_data(df):
    data = df.copy()

    data["Purchase_Month"] = (
        data["InvoiceDate"]
        .dt.to_period("M")
    )

    first_purchase = (
        data.groupby("CustomerID")["Purchase_Month"]
        .min()
        .rename("Cohort_Month")
    )

    data = data.merge(
        first_purchase,
        on="CustomerID",
        how="left"
    )

    data["Cohort_Period"] = (
        (data["Purchase_Month"].dt.year -
         data["Cohort_Month"].dt.year) * 12
        +
        (data["Purchase_Month"].dt.month -
         data["Cohort_Month"].dt.month)
        + 1
    )

    return data


def build_retention_table(df):
    cohort_counts = (
        df.groupby(
            ["Cohort_Month", "Cohort_Period"]
        )["CustomerID"]
        .nunique()
        .reset_index()
    )

    cohort_pivot = cohort_counts.pivot(
        index="Cohort_Month",
        columns="Cohort_Period",
        values="CustomerID"
    )

    cohort_size = cohort_pivot.iloc[:, 0]

    return cohort_pivot.divide(
        cohort_size,
        axis=0
    ) * 100


def route_query(query):
    q = query.lower()

    if (
        "revenue" in q
        and (
            "fall" in q
            or "drop" in q
            or "trend" in q
            or "growth" in q
            or "change" in q
        )
    ):
        return "revenue_trend"

    if (
        "retention" in q
        or "cohort" in q
        or "retained" in q
        or "repeat purchase" in q
    ):
        return "cohort"

    if (
        "customer" in q
        or "segment" in q
        or "at risk" in q
        or "champion" in q
        or "rfm" in q
    ):
        return "rfm"

    return "unknown"


def get_data(intent, data_contract):
    if intent == "revenue_trend":
        return data_contract["revenue_trend"]

    if intent == "rfm":
        rfm = data_contract["rfm_customers"]

        return (
            rfm[rfm["Segment"] == "At Risk"]
            .sort_values("Monetary", ascending=False)
            .head(20)
        )

    if intent == "cohort":
        return data_contract["retention"]

    return None


def narrate(query, intent, data):
    if intent == "revenue_trend":
        context = """
The data contains monthly revenue, order count, customer count,
and revenue growth information.
Use this to explain revenue performance and changes over time.
"""

    elif intent == "rfm":
        context = """
The data contains customer Recency, Frequency, Monetary values,
RFM scores, and customer segments.
Use this to explain customer behavior and identify valuable or
at-risk customer groups.
"""

    elif intent == "cohort":
        context = """
The data is a cohort retention table.
Rows represent customer cohorts based on their first purchase month.
Columns represent months since the customer's first purchase.
Values are retention percentages.
Use this to explain customer retention patterns over time.
"""

    else:
        context = "Explain the provided business data accurately."

    prompt = f"""
You are a business analyst assistant.

The user asked:
"{query}"

Type of analysis:
{intent}

How to interpret the data:
{context}

Here is the exact computed data:
{data}

Do not invent, alter, or calculate unsupported numbers.
Only use information contained in the provided data.

Give a concise, business-focused explanation in 3-5 sentences.
"""

    api_key = os.environ.get("GEMINI_API_KEY")

    if not api_key:
        return (
            "The analytics were calculated successfully, "
            "but the AI explanation is unavailable because "
            "GEMINI_API_KEY is not configured."
        )

    try:
        client = genai.Client(api_key=api_key)

        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )

        return response.text

    except Exception as e:
        return (
            "The analytics were calculated successfully, "
            "but the AI explanation is temporarily unavailable. "
            f"Gemini error: {type(e).__name__}"
        )



Writing backend.py


## 9. NEXUS Streamlit application

This creates the dashboard and connects it to the shared backend/data contract.


In [ ]:
%%writefile app.py
import os
import streamlit as st
import pandas as pd
import plotly.express as px

from backend import (
    load_and_clean_data,
    build_revenue_data,
    build_rfm,
    build_cohort_data,
    build_retention_table,
    route_query,
    get_data,
    narrate
)


st.set_page_config(
    page_title="NEXUS — AI Business Intelligence",
    page_icon="✦",
    layout="wide",
    initial_sidebar_state="collapsed"
)

FILE_PATH = "/content/Online Retail.xlsx"


@st.cache_data
def load_data():
    return load_and_clean_data(FILE_PATH)


df_clean = load_data()

revenue_df = build_revenue_data(df_clean)
rfm_df = build_rfm(df_clean)

cohort_data = build_cohort_data(df_clean)
retention_df = build_retention_table(cohort_data)

data_contract = {
    "clean_transactions": df_clean,
    "revenue_trend": revenue_df,
    "rfm_customers": rfm_df,
    "retention": retention_df
}


st.markdown("""
<style>
.stApp {
    background:
        radial-gradient(circle at 10% 10%, rgba(99,102,241,0.12), transparent 30%),
        radial-gradient(circle at 90% 20%, rgba(168,85,247,0.10), transparent 30%),
        #08090d;
    color: #f5f5f7;
}

#MainMenu {visibility: hidden;}
footer {visibility: hidden;}
header {visibility: hidden;}

.block-container {
    max-width: 1450px;
    padding-top: 2.5rem;
    padding-bottom: 4rem;
}

.brand {
    font-size: 2rem;
    font-weight: 700;
    letter-spacing: -1px;
}

.brand-sub {
    color: #8b8d98;
    font-size: 0.78rem;
    letter-spacing: 2px;
    text-transform: uppercase;
}

.status {
    color: #7df2a6;
    font-size: 0.78rem;
    letter-spacing: 1px;
}

.status-dot {
    display: inline-block;
    width: 7px;
    height: 7px;
    border-radius: 50%;
    background: #7df2a6;
    margin-right: 7px;
}

.kpi {
    background: rgba(255,255,255,0.035);
    border: 1px solid rgba(255,255,255,0.075);
    border-radius: 18px;
    padding: 1.4rem 1.5rem;
    min-height: 125px;
}

.kpi-title {
    color: #858793;
    font-size: 0.75rem;
    text-transform: uppercase;
    letter-spacing: 1.5px;
}

.kpi-value {
    font-size: 2rem;
    font-weight: 600;
    margin-top: 10px;
}

.kpi-change {
    color: #7df2a6;
    font-size: 0.78rem;
    margin-top: 5px;
}

.panel {
    background: rgba(255,255,255,0.035);
    border: 1px solid rgba(255,255,255,0.075);
    border-radius: 20px;
    padding: 1.5rem;
}

.section-label {
    color: #777985;
    font-size: 0.72rem;
    font-weight: 600;
    letter-spacing: 2px;
    text-transform: uppercase;
    margin-bottom: 1rem;
}

.ai-card {
    background:
        linear-gradient(
            135deg,
            rgba(99,102,241,0.14),
            rgba(168,85,247,0.06)
        );
    border: 1px solid rgba(139,92,246,0.25);
    border-radius: 20px;
    padding: 1.5rem;
}
</style>
""", unsafe_allow_html=True)


# Header
col1, col2 = st.columns([4, 1])

with col1:
    st.markdown("""
    <div class="brand">NEXUS</div>
    <div class="brand-sub">AI Business Intelligence</div>
    """, unsafe_allow_html=True)

with col2:
    st.markdown("""
    <div class="status">
    <span class="status-dot"></span>
    SYSTEM ONLINE
    </div>
    """, unsafe_allow_html=True)


st.divider()


# Business pulse
st.markdown(
    '<div class="section-label">Business Pulse</div>',
    unsafe_allow_html=True
)

total_revenue = df_clean["Revenue"].sum()
total_orders = df_clean["InvoiceNo"].nunique()
total_customers = df_clean["CustomerID"].nunique()

col1, col2, col3 = st.columns(3)

with col1:
    st.markdown(f"""
    <div class="kpi">
        <div class="kpi-title">Total Revenue</div>
        <div class="kpi-value">${total_revenue:,.0f}</div>
        <div class="kpi-change">Calculated from dataset</div>
    </div>
    """, unsafe_allow_html=True)

with col2:
    st.markdown(f"""
    <div class="kpi">
        <div class="kpi-title">Orders</div>
        <div class="kpi-value">{total_orders:,}</div>
        <div class="kpi-change">Unique invoices</div>
    </div>
    """, unsafe_allow_html=True)

with col3:
    st.markdown(f"""
    <div class="kpi">
        <div class="kpi-title">Customers</div>
        <div class="kpi-value">{total_customers:,}</div>
        <div class="kpi-change">Unique customers</div>
    </div>
    """, unsafe_allow_html=True)


st.divider()


# Real charts
left, right = st.columns([1.5, 1])

with left:
    st.markdown(
        '<div class="section-label">Revenue Trajectory</div>',
        unsafe_allow_html=True
    )

    fig = px.line(
        revenue_df,
        x="Month",
        y="Revenue",
        markers=True
    )

    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=10, r=10, t=20, b=10),
        height=350
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

with right:
    st.markdown(
        '<div class="section-label">Customer Health</div>',
        unsafe_allow_html=True
    )

    segment_counts = (
        rfm_df["Segment"]
        .value_counts()
        .reset_index()
    )

    segment_counts.columns = [
        "Segment",
        "Customers"
    ]

    fig = px.pie(
        segment_counts,
        names="Segment",
        values="Customers",
        hole=0.55
    )

    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=10, r=10, t=20, b=10),
        height=350
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )


st.divider()


# AI Analyst
st.markdown(
    '<div class="section-label">Ask the Analyst</div>',
    unsafe_allow_html=True
)

query = st.text_input(
    "Business question",
    placeholder="Why did revenue fall?  •  Which customers are at risk?  •  How is customer retention?"
)

if st.button("✦ Analyze", use_container_width=False):

    if not query.strip():
        st.warning("Enter a business question first.")

    else:
        intent = route_query(query)

        if intent == "unknown":
            st.warning(
                "I can currently analyze revenue trends, customer/RFM risk, "
                "and cohort retention."
            )

        else:
            data = get_data(
                intent,
                data_contract
            )

            with st.spinner("Analyzing business data..."):
                answer = narrate(
                    query,
                    intent,
                    data
                )

            st.markdown("""
            <div class="ai-card">
                <strong>✦ AI Executive Insight</strong>
            </div>
            """, unsafe_allow_html=True)

            st.write(answer)

            with st.expander("Show analysis type"):
                st.write(f"Intent: `{intent}`")
                st.dataframe(data, use_container_width=True)



Writing app.py


## 10. Final Day 2 verification

This should be the only notebook output you need before launching NEXUS.


In [ ]:
print("============================================================")
print("DAY 2 INTEGRATION CHECK")
print("============================================================")
print("Clean transactions:", df_clean.shape)
print("Revenue periods:   ", revenue_df.shape)
print("RFM customers:     ", rfm_df.shape)
print("Cohort table:      ", retention_df.shape)
print("Elasticity rows:   ", elasticity_df.shape)
print("Product actions:   ", discontinuation_df.shape)
print("Data contract keys:", list(data_contract.keys()))
print("============================================================")
print("backend.py created: True")
print("app.py created:     True")
print("============================================================")


DAY 2 INTEGRATION CHECK
Clean transactions: (392692, 12)
Revenue periods:    (13, 4)
RFM customers:      (4338, 8)
Cohort table:       (13, 13)
Elasticity rows:    (2567, 8)
Product actions:    (3897, 11)
Data contract keys: ['clean_transactions', 'revenue_trend', 'product_profitability', 'rfm_customers', 'retention', 'gross_margin', 'price_elasticity', 'product_discontinuation']
backend.py created: True
app.py created:     True


## 11. Launch NEXUS

Run this cell after the verification cell succeeds.

If port `8501` is already occupied, Streamlit will automatically choose another available port.


In [ ]:
!streamlit run app.py &>/content/streamlit.log &

import time
time.sleep(3)

print(open("/content/streamlit.log").read()[-2500:])




2026-09-04 21:17:58.592 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.106.195.187:8501




12. Ngrok Interface

In [ ]:
!pip install -q pyngrok

In [ ]:
!streamlit run app.py --server.port 8501 > /content/streamlit.log 2>&1 &

In [ ]:
import time
time.sleep(5)

!cat /content/streamlit.log



2026-09-04 21:51:54.353 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.106.195.187:8501



In [ ]:
!ps aux | grep "[s]treamlit"

root       18708  4.8  0.5 451848 71784 ?        Sl   21:51   0:03 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.port 8501


In [ ]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501)

print("NEXUS URL:")
print(public_url)

NEXUS URL:
NgrokTunnel: "https://unwind-untrained-curdle.ngrok-free.dev" -> "http://localhost:8501"
